In [1]:
!pip install kagglehub
!pip install langdetect
!pip install transformers
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=ba1c27d16c5150d77277faa60818f813a24db6b9d0d012dbb3c710d5d131f02c
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 4.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12

In [2]:
# Disable W&B logging for clean ouput
import os
os.environ["WANDB_DISABLED"] = "true"
import kagglehub
import numpy as np
import pandas as pd
from langdetect import detect
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer,BertForSequenceClassification, BertModel, Trainer, TrainingArguments
# from torch.utils.data import Dataset
from datasets import DatasetDict, Dataset
from sklearn.metrics import accuracy_score
import torch.nn as nn
import torch
from sklearn.utils.class_weight import compute_class_weight

2025-05-08 21:22:50.589282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746739370.773301      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746739370.827928      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Download dataset
def load_dataset(used_column):
    dataset_path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")
    print("dataset downloaded to this path:", dataset_path)
    ds = pd.read_csv(os.path.join(dataset_path,'aa_dataset-tickets-multi-lang-5-2-50-version.csv'),usecols=used_column)
    ds = ds.rename(columns={'queue': 'label'})
    return ds

In [4]:
# Retain necessary columns
used_column = ['body','queue'] #['type','queue','priority']
ds = load_dataset(used_column)

# Check null label value, and discard the value
if ds['label'].isnull().any():
    print('There are some rows that has null label value, discard the rows')
    ds = ds.dropna(subset=['label'])

# We use the langdetect library to label the language used in the text
ds['lang'] = ds['body'].apply(lambda x: detect(str(x)))

dataset downloaded to this path: /kaggle/input/multilingual-customer-support-tickets


In [5]:
# Discard non english text
ds = ds[ds['lang'] == 'en']
ds = ds.drop(columns=['lang']).reset_index(drop=True)

# Display labels
print('Unique value for category:', ds['label'].unique())
# Check label distribution
print('Label freq:', ds['label'].value_counts(normalize=True) * 100)
# Display total row number for each label
print('Label freq:', ds['label'].value_counts())

# Enumerate categories
label2id = {label: idx for idx, label in enumerate(ds['label'].unique())}
ds['label'] = ds['label'].map(label2id)

# reverse
# id2label = {v: k for k, v in label2id.items()}

Unique value for category: ['Technical Support' 'Returns and Exchanges' 'Billing and Payments'
 'Sales and Pre-Sales' 'Service Outages and Maintenance' 'Product Support'
 'IT Support' 'Customer Service' 'Human Resources' 'General Inquiry']
Label freq: label
Technical Support                  29.405461
Product Support                    18.514927
Customer Service                   14.605767
IT Support                         12.054095
Billing and Payments                9.762695
Returns and Exchanges               5.103343
Service Outages and Maintenance     4.067364
Sales and Pre-Sales                 3.041592
Human Resources                     2.051544
General Inquiry                     1.393213
Name: proportion, dtype: float64
Label freq: label
Technical Support                  5762
Product Support                    3628
Customer Service                   2862
IT Support                         2362
Billing and Payments               1913
Returns and Exchanges              1000
S

In [6]:
ds_ready = ds.copy()
# Split the dataset
train_ds, test_ds = train_test_split(ds_ready, test_size=0.2, random_state=42, stratify=ds_ready['label'])
# Convert to Huggingface dataset object
train_dataset = Dataset.from_pandas(train_ds.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_ds.reset_index(drop=True))
ready_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [7]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(datas):
    return tokenizer(datas["body"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = ready_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/15676 [00:00<?, ? examples/s]

Map:   0%|          | 0/3919 [00:00<?, ? examples/s]

In [8]:
num_labels = train_ds['label'].nunique()
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# # Get class weights based on label column
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(train_ds['label']),
#     y=train_ds['label']
# )

# # Convert to tensor for PyTorch
# class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# class WeightedDistilBERT(DistilBertForSequenceClassification):
#     def __init__(self, config, class_weights):
#         super().__init__(config)
#         self.class_weights = class_weights

#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.get("labels")
#         outputs = model(**inputs)
#         logits = outputs.get("logits")
#         loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
#         loss = loss_fct(logits, labels)
#         return (loss, outputs) if return_outputs else loss

# # Config with correct number of labels
# config = DistilBertConfig.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

# # Initialize model with custom weights
# model = WeightedDistilBERT(config, class_weights_tensor)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [9]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,roc_auc_score

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)  # Get class with max logit
    labels = pred.label_ids
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    # Multiclass ROC AUC (macro average, One-vs-Rest)
    try:
        roc_auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        # Occurs when only one class present in a batch (e.g., small val set)
        roc_auc = float("nan")

    return {
        'accuracy': acc,
        'macro_f1': f1,
        'macro_precision': precision,
        'macro_recall': recall,
        "roc_auc": roc_auc
    }

In [13]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.07,
    learning_rate=5e-5, 
    logging_dir="./logs",
    logging_steps=10,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train the frozen model — only the classification head will be updated
print("\n🚀 Fine-tuning (frozen BERT encoder)...")
trainer.train()

# Evaluate the model after training with frozen BERT layers
print("\n📊 Evaluating frozen encoder model...")
trainer_results = trainer.evaluate()       # Evaluate on test set
print(trainer_results)
#frozen_acc = frozen_results["eval_accuracy"]     # Extract accuracy


🚀 Fine-tuning (frozen BERT encoder)...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,0.271700,1.452002,0.639449,0.575928,0.651568,0.552463
2,0.165300,1.616892,0.641235,0.620326,0.668686,0.597351
3,0.096700,1.845842,0.664200,0.636577,0.715533,0.609937
4,0.087700,1.816704,0.690482,0.678460,0.703998,0.660169
5,0.057700,1.806642,0.702730,0.692386,0.716716,0.673401


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



📊 Evaluating frozen encoder model...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 1.8066415786743164, 'eval_accuracy': 0.702730288338862, 'eval_macro_f1': 0.6923855477233889, 'eval_macro_precision': 0.7167162517581596, 'eval_macro_recall': 0.673401383174556, 'eval_runtime': 20.2338, 'eval_samples_per_second': 193.686, 'eval_steps_per_second': 6.079, 'epoch': 5.0}
